In [1]:
# =========================================================================
# PURPOSE
#   Estimate Bayesian impulse response functions (IRFs) of an
#   Autoregressive Distributed Lag (ADL) model with a *constant*
#   residual variance and a *flat* (improper, uninformative) prior on
#   the regression coefficients. The shock variable z_t is split into
#   its positive (z_t^+) and negative parts (z_t^-) so that asymmetric responses to
#   "good news" and "bad news" can be measured separately.
#
#   Model (one combination y, z, p, q):
#       y_t = alpha
#             + sum_{j=0}^{q} ( beta_j^+ * z_t^+ + beta_j^- * z_t^- )_{t-j}
#             + sum_{i=1}^{p} gamma_i * y_{t-i}
#             + u_t,                              u_t ~ N(0, sigma^2)
#
#       z_t^+ = max(z_t, 0)        (positive component of the shock)
#       z_t^- = -min(z_t, 0)       (positive image of the negative part)
#
#   Posterior draws are obtained with a Gibbs sampler that alternates
#   between:
#     - sigma^2 | beta, y ~ Inverse-Gamma (T/2, SSR/2),
#     - beta    | sigma^2, y ~ Normal with covariance (H + X'X/sigma^2)^-1
#       and mean (H + X'X/sigma^2)^-1 (H mu + X'y/sigma^2),
#       where (H, mu) describe the Gaussian prior. With the flat prior
#       used here H = 0 and the conditional collapses to the standard
#       Bayesian linear regression with constant variance.
#
#   Each draw is rejected and re-drawn (up to 500 attempts) if the
#   companion matrix associated with the AR coefficients on y has roots
#   outside the unit circle, i.e. if it would generate an explosive AR
#   process.
#
#   External dependencies:
#       - Data.xls (input data; column 0 is a Date that is dropped)
# =========================================================================

get_ipython().system('pip install xlrd')

import numpy as np
import pandas as pd
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
warnings.filterwarnings("ignore")
from Functions import check_stab, run_irf_constvar, flat_prior
np.random.seed(42)

# -------------------------------------------------------------------------
#    Load the data set
#    The first column of the spreadsheet is the date (which is dropped),
#    the remaining columns contain {EUA, ESG, Supply, Demand, Risk}.
# -------------------------------------------------------------------------
df   = pd.read_excel(
    '/Users/johannesfelchner/Desktop/Code/Section 6/Section 6.1/Section 6.1.1/Data.xls',
    header=None,
)
data = df.iloc[:, 1:].values.astype(float)

#    The six (yc, zc, p, q) tuples to estimate.
#     yc / zc are column indices into `data`:
#       0 -> EUA, 1 -> ESG, 2 -> Supply, 3 -> Demand, 4 -> Risk.
combos = [
    (0, 2, 5,  1),    # EUA  reacting to Supply shock
    (0, 3, 5,  0),    # EUA  reacting to Demand shock
    (0, 4, 5,  0),    # EUA  reacting to Risk   shock
    (1, 2, 10, 1),    # ESG  reacting to Supply shock
    (1, 3, 9,  1),    # ESG  reacting to Demand shock
    (1, 4, 10, 0),    # ESG  reacting to Risk   shock
]

#    MCMC and IRF settings.
Reps    = 60000      # total Gibbs iterations
burn    = 20000      # burn-in iterations
Nirf    = 20         # IRF horizon (in periods)
prior_t = 10         # prior tightness placeholder (unused with flat prior)

# -------------------------------------------------------------------------
#    Run the sampler on each (y, z, p, q) combination
# -------------------------------------------------------------------------
results = []
for idx, (yc, zc, p, q) in enumerate(combos):
    lb = ["EUA×Sup", "EUA×Dem", "EUA×Risk", "ESG×Sup", "ESG×Dem", "ESG×Risk"]
    print(f"  {idx+1}/6 {lb[idx]} (p={p},q={q})")
    results.append(run_irf_constvar(data[:, yc], data[:, zc], p, q,Reps,burn,Nirf,prior_t))

# -------------------------------------------------------------------------
#    Plotting: 3 x 4 panel of IRFs
# -------------------------------------------------------------------------
#     Colour palette.
cLine  = '#1B5E8A'        # posterior median
cFill  = '#B0CEE4'        # 16-84 credible band fill
cEdge  = '#6FA3C6'        # band edges
cZero  = '#999999'        # zero line
cPanel = '#FFFFFF'        # panel background

#    Global matplotlib style.
plt.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset':  'dejavuserif',
    'font.size':         7,
    'axes.linewidth':    0.4,
    'axes.edgecolor':    '#BBBBBB',
    'xtick.major.width': 0.35, 'ytick.major.width': 0.35,
    'xtick.major.size':  2.2,  'ytick.major.size':  2.2,
    'xtick.direction':   'out', 'ytick.direction':  'out',
    'xtick.color':       '#555555', 'ytick.color':  '#555555',
    'axes.labelcolor':   '#222222', 'text.color':   '#222222',
})

fig = plt.figure(figsize=(7.48, 7.0), facecolor='white')

#     4-row outer grid: top header strip + 3 shock-name rows.
outer = gridspec.GridSpec(
    4, 1, figure=fig, height_ratios=[0.06, 1, 1, 1],
    left=0.09, right=0.965, top=0.97, bottom=0.07,
    hspace=0.14,
)

#     Top header: asset names spanning the four columns.
ax_h = fig.add_subplot(outer[0])
ax_h.set_xlim(0, 1); ax_h.set_ylim(0, 1); ax_h.axis('off')
ax_h.text(0.235, 0.85, 'EUA Futures Returns', fontsize=8.5,
          fontweight='bold', ha='center', va='center', color='#1a1a1a')
ax_h.text(0.735, 0.85, 'ESG Index Returns',        fontsize=8.5,
          fontweight='bold', ha='center', va='center', color='#1a1a1a')

x = np.arange(Nirf)
shock_names = ['News Supply Shock', 'News Demand Shock', 'News Risk Shock']

all_axes = []                          # collected for later y-axis matching

#     Iterate over the three shock rows.
for row in range(3):
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 4, subplot_spec=outer[row + 1], wspace=0.40,
    )
    eua = results[row]
    esg = results[row + 3]
    _, _, pe, qe = combos[row]
    _, _, ps, qs = combos[row + 3]

    #     The four panels in this row, in the order
    #       (EUA+, EUA-, ESG+, ESG-).
    panels_data = [
        (eua[0], eua[1], eua[2]), (eua[3], eua[4], eua[5]),
        (esg[0], esg[1], esg[2]), (esg[3], esg[4], esg[5]),
    ]

    #      For Supply (row 0) and Risk (row 2) we swap the +/- columns
    #        so that the same economic interpretation appears in the
    #        same screen position throughout the figure. The data itself
    #        is unchanged; only the display position is permuted.
    if row in (0, 2):
        panels_data = [panels_data[1], panels_data[0],
                       panels_data[3], panels_data[2]]

    for ci, (med, up, lo) in enumerate(panels_data):
        ax = fig.add_subplot(inner[ci])
        ax.set_facecolor(cPanel)
        ax.fill_between(x, lo, up, color=cFill, alpha=0.5,
                        linewidth=0, zorder=2)
        ax.plot(x, up,  color=cEdge, linewidth=0.4,  zorder=3)
        ax.plot(x, lo,  color=cEdge, linewidth=0.4,  zorder=3)
        ax.plot(x, med, color=cLine, linewidth=1.05, zorder=4,
                solid_capstyle='round')
        ax.axhline(0, color=cZero, linewidth=0.4, zorder=1)

        # Cosmetic axis tweaks.
        ax.set_xlim(-0.3, Nirf - 0.7)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
        ax.tick_params(labelsize=5.8, pad=1.5)
        for sp in ['top', 'right']:
            ax.spines[sp].set_visible(False)
        ax.yaxis.grid(True, linewidth=0.25, color='#E0E0E0', zorder=0)
        ax.xaxis.grid(False)
        ax.set_axisbelow(True)

        # In-panel annotation: the (p, q) used for this combination.
        pq = f'p = {pe}, q = {qe}' if ci < 2 else f'p = {ps}, q = {qs}'
        ax.text(0.97, 0.95, pq, transform=ax.transAxes,
                fontsize=5.5, va='top', ha='right',
                color='#666666', fontstyle='italic')

        # Top row gets the +/- sub-titles.
        if row == 0:
            sub_title = 'Positive News Shock' if ci % 2 == 0 \
                                              else 'Negative News Shock'
            ax.set_title(sub_title, fontsize=7.5, fontstyle='italic',
                         color='#444444', pad=6)

        # Bottom row gets the x-label; first column gets the row label.
        if row == 2:
            ax.set_xlabel('Periods', fontsize=6.5, labelpad=2)
        if ci == 0:
            ax.set_ylabel(shock_names[row], fontsize=7.5,
                          fontweight='bold', labelpad=6)

        all_axes.append((row, ci, ax, med, up, lo))

# -------------------------------------------------------------------------
#    Match the y-axis range across the four panels of each row, and
#    pick a sensible number-format depending on the magnitude.
# -------------------------------------------------------------------------
for row in range(3):
    row_axes = [(ci, ax, med, up, lo)
                for r, ci, ax, med, up, lo in all_axes if r == row]

    global_max = 0
    for ci, ax, med, up, lo in row_axes:
        panel_max = max(abs(up.max()), abs(lo.min()),
                        abs(med.max()), abs(med.min()))
        global_max = max(global_max, panel_max)

    ylim = global_max * 1.08
    for ci, ax, med, up, lo in row_axes:
        ax.set_ylim(-ylim, ylim)
        if   ylim > 0.5:
            ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
        elif ylim > 0.05:
            ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
        else:
            ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))
        ax.yaxis.set_major_locator(
            ticker.MaxNLocator(nbins=5, symmetric=True),
        )

# -------------------------------------------------------------------------
#    Single shared legend at the bottom of the figure
# -------------------------------------------------------------------------
legend_handles = [
    Line2D([0], [0], color=cLine, linewidth=1.2,  label='Median'),
    Line2D([0], [0], color=cEdge, linewidth=0.55, label='16th / 84th Percentile'),
    plt.Rectangle((0, 0), 1, 1, fc=cFill, alpha=0.5, ec='none',
                  label='68 % Credible Interval'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=3,
           fontsize=6.5, frameon=False, bbox_to_anchor=(0.53, 0.005),
           handlelength=2.0, handletextpad=0.5, columnspacing=2.5)

# -------------------------------------------------------------------------
#    Save to disk (PDF for LaTeX inclusion)
# -------------------------------------------------------------------------

fig.savefig(
    '/Users/johannesfelchner/Desktop/Code/Section 6/Section 6.1/Section 6.1.1/IRF_Bayesian_Panel_flattest.pdf',
    bbox_inches='tight', facecolor='white', edgecolor='none',
)
print("IRF figure done.")


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
  1/6 EUA×Sup (p=5,q=1)
  2/6 EUA×Dem (p=5,q=0)
  3/6 EUA×Risk (p=5,q=0)
  4/6 ESG×Sup (p=10,q=1)
  5/6 ESG×Dem (p=9,q=1)
  6/6 ESG×Risk (p=10,q=0)
IRF figure done.
